# M-Lab India — k-means network-quality profiles

Groups the cleaned measurements (from `outlier.ipynb`) into a small number of representative
"network condition" profiles via k-means, run separately for download and upload. All the
machinery (transform, k sweep, k selection, cluster report, ranking, nearest-real-row lookup) is
defined once below and reused for both directions.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'mlab_analysis':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get('MLAB_OUTPUT_DIR', PROJECT_ROOT / 'outputs' / 'analysis' / '2026-06-18_to_2026-11-04'))
FIGURE_DIR = DATA_DIR / 'figures' / 'clustering'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# High-contrast, colour-blind-friendly plotting colours (Okabe-Ito inspired).
BLUE, ORANGE, GREY, MAGENTA, SKY_BLUE = '#0072B2', '#E69F00', '#7A7A7A', '#CC79A7', '#56B4E9'
RED, GREEN = MAGENTA, BLUE  # Backward-compatible semantic aliases used by helper plots.
RANDOM_STATE = 42

download = pd.read_csv(DATA_DIR / 'download_clean.csv', low_memory=False)
upload = pd.read_csv(DATA_DIR / 'upload_clean.csv', low_memory=False)
DOWNLOAD_FEATURES = ['download_mbps', 'min_rtt_ms', 'loss_rate']
UPLOAD_FEATURES = ['upload_mbps', 'min_rtt_ms']

print(f'Clean download: {download.shape}   Clean upload: {upload.shape}')

## Method (for the methodology slide)

1. **Transform** each feature with `log1p`. Throughput and RTT are heavily right-skewed (a few
   very fast/very slow measurements would otherwise dominate the Euclidean distance k-means uses);
   `log1p` compresses the long tail and is safe at zero (`log1p(0) = 0`, unlike plain `log`).
2. **Standardize** the log1p'd features with `StandardScaler` (zero mean, unit variance) so that a
   feature with a bigger numeric range (e.g. `min_rtt_ms`, ~0.2-5000+) doesn't dominate one with a
   smaller range (e.g. `loss_rate`, 0-1) just because of scale — k-means distance is scale-sensitive.
3. **Fit k-means for k = 2..6**, fixed `random_state=42` and `n_init=10` (10 random centroid seeds
   per k, keep the best by inertia) so results are reproducible.
4. **Score each k** two ways: `inertia` (within-cluster sum of squared distances — always drops as
   k grows, so it's read for its *bend*, not its minimum) and `silhouette score` (-1 to 1; how much
   tighter/better-separated a point's own cluster is vs. the next-best one — this one *can* be
   compared across k directly). Silhouette is computed on a fixed random 5,000-row subsample
   (`sample_size=5000, random_state=42`): silhouette needs all pairwise distances (O(n^2)), so a
   full ~27k-32k row sample would mean a huge distance matrix per k. The subsample is a standard,
   reproducible way to get the same statistic cheaply, without any GPU-specific libraries.
5. **Pick k** favoring the smallest k whose silhouette is within 0.01 of the best score in the
   sweep — a presentation with 3 clearly-separated profiles is more defensible and useful than one
   with 6 profiles that are only marginally better statistically.
6. **Rank profiles** by a composite z-scored score (throughput up is better, RTT/loss up is worse),
   not by throughput alone -- a fast-but-lossy cluster and a slow-but-clean cluster don't sort the
   same way on every metric, so a single-column sort can silently mis-rank them.

In [ ]:
def transform_and_scale(df, features):
    '''log1p each feature (fixes right skew, safe at 0), then z-score standardize.
    Returns (X_scaled, fitted scaler) -- the scaler lets us invert back to original units later.'''
    log_values = np.log1p(df[features])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(log_values)
    return X_scaled, scaler

def evaluate_k_range(X_scaled, k_range=range(2, 7), silhouette_sample_size=5000):
    rows, models = [], {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels,
                                sample_size=min(silhouette_sample_size, len(X_scaled)),
                                random_state=RANDOM_STATE)
        rows.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil})
        models[k] = km
    return pd.DataFrame(rows), models

def plot_k_selection(k_eval, dataset_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(k_eval['k'], k_eval['inertia'], marker='o', color=BLUE)
    axes[0].set(title=f'{dataset_name.title()}: elbow curve', xlabel='k', ylabel='Inertia (within-cluster SS)')
    axes[0].grid(alpha=.3)
    axes[1].plot(k_eval['k'], k_eval['silhouette'], marker='o', color=ORANGE)
    axes[1].set(title=f'{dataset_name.title()}: silhouette score vs. k', xlabel='k', ylabel='Silhouette score')
    axes[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f'{dataset_name}_k_selection.png', dpi=180); plt.show()

def choose_k(k_eval, tolerance=0.01):
    '''Smallest k whose silhouette is within `tolerance` of the best silhouette in the sweep --
    favors a simpler, more presentable clustering when scores are close, instead of always
    taking the statistical maximum.'''
    best_row = k_eval.loc[k_eval['silhouette'].idxmax()]
    candidates = k_eval[k_eval['silhouette'] >= best_row['silhouette'] - tolerance]
    chosen = int(candidates['k'].min())
    print(f"Best silhouette: k={int(best_row['k'])} (score={best_row['silhouette']:.3f})")
    print(f"Selected k = {chosen} (smallest k within {tolerance} of the best silhouette score)")
    return chosen

def build_cluster_report(df, features, km, scaler):
    labels = km.labels_
    working = df[features + ['measurement_id', 'client_asn', 'client_as_name', 'state']].copy()
    working['cluster'] = labels
    n = len(working)
    rows = []
    for c in range(km.n_clusters):
        sub = working[working['cluster'] == c]
        centroid_log = scaler.inverse_transform(km.cluster_centers_[c].reshape(1, -1))[0]
        centroid_orig = np.expm1(centroid_log)
        row = {'cluster': c, 'count': len(sub), 'percent': round(100 * len(sub) / n, 2)}
        for i, feature in enumerate(features):
            row[f'centroid_{feature}'] = centroid_orig[i]
            row[f'median_{feature}'] = sub[feature].median()
        rows.append(row)
    return pd.DataFrame(rows), working

def rank_profiles(report, throughput_col, profile_names, rtt_col='min_rtt_ms', loss_col=None):
    '''Composite quality score from z-scored MEDIANS (not centroids) across clusters:
    higher throughput is better (+), lower RTT is better (-), lower loss is better (-).
    A naive sort by throughput alone can rank clusters wrong when loss and throughput are not
    aligned (see the printed note below if that happens here) -- this is why a combined score
    is used instead of a single-column sort.'''
    report = report.copy()
    if len(profile_names) != len(report):
        raise ValueError('Provide exactly one descriptive name per cluster')
    def z(series):
        return (series - series.mean()) / series.std(ddof=0) if series.std(ddof=0) > 0 else series * 0
    score = z(report[f'median_{throughput_col}']) - z(report[f'median_{rtt_col}'])
    if loss_col:
        score = score - z(report[f'median_{loss_col}'])
    report['quality_score'] = score
    report = report.sort_values('quality_score', ascending=False).reset_index(drop=True)
    report['profile'] = profile_names[:len(report)]
    single_axis_order = report.sort_values(f'median_{throughput_col}', ascending=False)['cluster'].tolist()
    composite_order = report.sort_values('quality_score', ascending=False)['cluster'].tolist()
    if single_axis_order != composite_order:
        print('Note: ranking by throughput alone would have ordered clusters differently '
              f'({single_axis_order}) than the throughput+RTT+loss composite used here '
              f'({composite_order}) -- at least one cluster trades speed for materially worse loss.')
    return report

def nearest_rows_to_centroids(X_scaled, km, df):
    '''For each cluster, the actual observed row closest (Euclidean, in the same scaled/log1p
    space the model was fit on) to that cluster centroid -- a real measurement_id you can point
    to instead of only a synthetic centroid.'''
    nearest = []
    for c in range(km.n_clusters):
        idx_in_cluster = np.where(km.labels_ == c)[0]
        dists = np.linalg.norm(X_scaled[idx_in_cluster] - km.cluster_centers_[c], axis=1)
        best_idx = idx_in_cluster[np.argmin(dists)]
        nearest.append({'cluster': c, **df.iloc[best_idx].to_dict()})
    return pd.DataFrame(nearest)

def worst_profile_composition(report, working, top_n=8):
    '''Top ASNs and states inside the lowest-ranked (worst-quality) cluster -- a starting point
    for "what do we do about it": is the worst profile MORE concentrated on a few ISPs/regions than
    the dataset as a whole (actionable -- investigate that operator/area specifically) or does it
    mirror the overall carrier mix (systemic -- an access-technology/congestion pattern spread
    across the same major carriers as everyone else, not one bad network)? Concentration is judged
    relative to the overall dataset's top-3-ASN share, not an absolute threshold, since a couple of
    carriers (Jio, Airtel) dominate total row counts everywhere regardless of quality tier.'''
    worst_cluster = report.iloc[-1]['cluster']
    worst_profile = report.iloc[-1]['profile']
    sub = working[working['cluster'] == worst_cluster]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    asn_counts = sub['client_as_name'].fillna('<missing>').value_counts().head(top_n).sort_values()
    axes[0].barh(asn_counts.index, asn_counts.values, color=RED)
    axes[0].set(title=f'"{worst_profile}" profile: top ASNs (n={len(sub):,})', xlabel='Rows')
    axes[0].grid(axis='x', alpha=.25)
    state_counts = sub['state'].fillna('<missing>').value_counts().head(top_n).sort_values()
    axes[1].barh(state_counts.index, state_counts.values, color=RED)
    axes[1].set(title=f'"{worst_profile}" profile: top states', xlabel='Rows')
    axes[1].grid(axis='x', alpha=.25)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f'worst_profile_composition_{worst_profile.lower().replace(" ", "_")}.png',
                                     dpi=180, bbox_inches='tight'); plt.show()

    cluster_top3_share = sub['client_as_name'].value_counts(normalize=True).head(3).sum()
    overall_top3_share = working['client_as_name'].value_counts(normalize=True).head(3).sum()
    verdict = ('MORE concentrated than the overall mix -- worth investigating those specific operators/regions'
               if cluster_top3_share > overall_top3_share + 0.05
               else 'about the same concentration as the overall carrier mix -- likely a systemic '
                    'access-technology/congestion pattern, not one bad operator')
    print(f'"{worst_profile}": {100*len(sub)/len(working):.1f}% of rows, {sub["client_as_name"].nunique()} '
          f'distinct ASNs, {sub["state"].nunique()} states.')
    print(f'Top-3-ASN share inside this profile: {100*cluster_top3_share:.1f}%  '
          f'vs. {100*overall_top3_share:.1f}% overall -> {verdict}.')

## Cluster scatter-plot helpers

Colours below represent the K-means **cluster assignment** (mapped to the ranked profile name). The selected model's silhouette score is printed in the figure title, and stars mark the fitted centroids. K-means was fitted in standardized `log1p` space, even when a chart displays original units.

In [ ]:
def ranked_cluster_colours(report):
    palettes = {2: [BLUE, ORANGE],
                3: [BLUE, ORANGE, MAGENTA],
                4: [BLUE, SKY_BLUE, ORANGE, MAGENTA],
                5: [BLUE, SKY_BLUE, '#F0E442', ORANGE, MAGENTA],
                6: [BLUE, SKY_BLUE, '#009E73', '#F0E442', ORANGE, MAGENTA]}
    colours = palettes[len(report)]
    return {int(row.cluster): (row.profile, colours[i])
            for i, row in report.reset_index(drop=True).iterrows()}

def selected_silhouette(k_eval, chosen_k):
    return float(k_eval.loc[k_eval['k'].eq(chosen_k), 'silhouette'].iloc[0])

def scatter_one_pair(ax, frame, labels, report, x, y, title, xscale='linear', yscale='linear',
                     centroid_space='original', km=None):
    mapping = ranked_cluster_colours(report)
    for cluster, (profile, colour) in mapping.items():
        mask = np.asarray(labels) == cluster
        ax.scatter(frame.loc[mask, x], frame.loc[mask, y], s=8, alpha=.24, color=colour,
                   edgecolors='none', rasterized=True,
                   label=f'{profile} (n={mask.sum():,})')
        row = report.loc[report['cluster'].eq(cluster)].iloc[0]
        if centroid_space == 'original':
            cx, cy = row[f'centroid_{x}'], row[f'centroid_{y}']
        else:
            cx, cy = km.cluster_centers_[cluster, 0], km.cluster_centers_[cluster, 1]
        ax.scatter(cx, cy, marker='*', s=230, color=colour, edgecolor='black', linewidth=.9, zorder=5)
    ax.set(xlabel=x, ylabel=y, title=title, xscale=xscale, yscale=yscale)
    ax.grid(alpha=.18, which='both')
    ax.legend(fontsize=8, markerscale=2, loc='best')


## Download

Features: `download_mbps`, `min_rtt_ms`, `loss_rate`.

In [ ]:
download_X, download_scaler = transform_and_scale(download, DOWNLOAD_FEATURES)
download_k_eval, download_models = evaluate_k_range(download_X)
display(download_k_eval.round(4))
plot_k_selection(download_k_eval, 'download')
download_k = choose_k(download_k_eval)

In [ ]:
download_km = download_models[download_k]
download_report, download_working = build_cluster_report(download, DOWNLOAD_FEATURES, download_km, download_scaler)
download_report = rank_profiles(
    download_report,
    throughput_col='download_mbps',
    loss_col='loss_rate',
    profile_names=('Higher-performing', 'Throughput-constrained', 'Loss-affected'),
)
display(download_report.round(4))

### Download: three pairwise cluster views

With three download clustering features there are exactly three unique two-axis combinations. Loss uses a symmetric-log scale so that genuine zero-loss measurements remain visible.

In [ ]:
download_silhouette = selected_silhouette(download_k_eval, download_k)
fig, axes = plt.subplots(1, 3, figsize=(20, 5.7))
download_pairs = [
    ('download_mbps', 'min_rtt_ms', 'Throughput vs RTT', 'log', 'log'),
    ('download_mbps', 'loss_rate', 'Throughput vs loss', 'log', 'symlog'),
    ('min_rtt_ms', 'loss_rate', 'RTT vs loss', 'log', 'symlog'),
]
for ax, (x, y, title, xs, ys) in zip(axes, download_pairs):
    scatter_one_pair(ax, download, download_km.labels_, download_report, x, y, title, xs, ys)
    if y == 'loss_rate':
        ax.set_yscale('symlog', linthresh=1e-4)
        ax.set_ylim(bottom=0)
fig.suptitle(f'Download K-means assignments — k={download_k}, silhouette={download_silhouette:.3f}', y=.99, fontsize=15)
fig.tight_layout(rect=[0, 0, 1, .93]); fig.savefig(FIGURE_DIR / 'download_cluster_pairwise_scatter.png', dpi=200, bbox_inches='tight'); plt.show()

### Final presentation table — Download

In [ ]:
download_presentation = download_report[['profile', 'median_download_mbps', 'median_min_rtt_ms',
                                          'median_loss_rate', 'count', 'percent']].rename(columns={
    'median_download_mbps': 'Throughput (Mbps)', 'median_min_rtt_ms': 'RTT (ms)',
    'median_loss_rate': 'Loss (fraction)', 'count': 'Count', 'percent': 'Percentage', 'profile': 'Profile'})
download_presentation = download_presentation.round(
    {'Throughput (Mbps)': 2, 'RTT (ms)': 2, 'Loss (fraction)': 4, 'Percentage': 2})
download_presentation.to_csv(DATA_DIR / 'download_cluster_profiles.csv', index=False)
display(download_presentation)

download_colour_map = ranked_cluster_colours(download_report)
colors = [download_colour_map[int(cluster)][1] for cluster in download_report['cluster']]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(download_report['profile'], download_report['percent'], color=colors)
for y, (pct, mbps) in enumerate(zip(download_report['percent'], download_report['median_download_mbps'])):
    ax.text(pct + 0.5, y, f'{pct:.1f}%  (median {mbps:.1f} Mbps)', va='center', fontsize=9)
ax.set(title='Download network-quality profiles', xlabel='% of clean rows')
ax.invert_yaxis(); ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'download_cluster_profiles.png', dpi=180, bbox_inches='tight'); plt.show()

### Nearest real measurement to each centroid — Download

In [ ]:
download_nearest = nearest_rows_to_centroids(download_X, download_km, download)
download_nearest = download_nearest.merge(download_report[['cluster', 'profile']], on='cluster')
download_nearest_display = download_nearest[['profile', 'measurement_id', 'download_mbps', 'min_rtt_ms', 'loss_rate']]
download_nearest_display.to_csv(DATA_DIR / 'download_cluster_nearest_rows.csv', index=False)
display(download_nearest_display)

### Loss-affected profile — is it actionable?

The lowest-ranked download profile is not the slowest one. The **Throughput-constrained** profile
has about 7 Mbps median throughput, while the **Loss-affected** profile has about 20 Mbps but
approximately 15.85% loss. These names describe the dominant measured limitation rather than
claiming an absolute quality grade. The chart below checks whether the loss-affected profile is
concentrated on a few operators/regions or spread across the sample.

In [ ]:
worst_profile_composition(download_report, download_working)

### Download: where each profile is located

Simple India map, coloured by K-means profile assignment — no density shading, no zoom, just where
each network-quality profile actually shows up geographically.

In [ ]:
download_geo = download.loc[download_working.index, ['latitude', 'longitude']].copy()
download_geo['cluster'] = download_working['cluster'].values

fig, ax = plt.subplots(figsize=(8, 7))
for cluster, (profile, colour) in ranked_cluster_colours(download_report).items():
    sub = download_geo.loc[download_geo['cluster'] == cluster]
    ax.scatter(sub['longitude'], sub['latitude'], s=6, alpha=.35, color=colour, edgecolors='none',
               label=f'{profile} (n={len(sub):,})')
ax.set(xlim=(67, 99), ylim=(6, 38), title='Download: network-quality profile by location',
       xlabel='Longitude', ylabel='Latitude')
ax.grid(alpha=.15)
ax.legend(fontsize=9, markerscale=3, loc='lower left')
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'download_profile_map.png', dpi=180, bbox_inches='tight'); plt.show()

## Upload

Features: `upload_mbps`, `min_rtt_ms` (no loss feature — upload loss was dropped in `outlier.ipynb`
as all-null metadata, see that notebook's Section 2/schema step).

In [ ]:
upload_X, upload_scaler = transform_and_scale(upload, UPLOAD_FEATURES)
upload_k_eval, upload_models = evaluate_k_range(upload_X)
display(upload_k_eval.round(4))
plot_k_selection(upload_k_eval, 'upload')
upload_k = choose_k(upload_k_eval)

In [ ]:
upload_km = upload_models[upload_k]
upload_report, upload_working = build_cluster_report(upload, UPLOAD_FEATURES, upload_km, upload_scaler)
upload_report = rank_profiles(
    upload_report,
    throughput_col='upload_mbps',
    profile_names=('Higher-performing', 'Throughput-constrained'),
)
display(upload_report.round(4))

### Upload: three complementary two-axis cluster views

Upload has only two clustering features, so only one unique feature pair exists. To provide three useful views without inventing a third feature: (1) original units on log axes show the full tails, (2) a central-99% linear zoom shows overlap near the dense region, and (3) standardized `log1p` model space shows the geometry K-means actually used. All three show the same assignments.

In [ ]:
upload_silhouette = selected_silhouette(upload_k_eval, upload_k)
fig, axes = plt.subplots(1, 3, figsize=(20, 5.7))
scatter_one_pair(axes[0], upload, upload_km.labels_, upload_report,
                 'upload_mbps', 'min_rtt_ms', 'Full range: original units', 'log', 'log')

scatter_one_pair(axes[1], upload, upload_km.labels_, upload_report,
                 'upload_mbps', 'min_rtt_ms', 'Central 99%: original units')
axes[1].set_xlim(0, upload['upload_mbps'].quantile(.995))
axes[1].set_ylim(0, upload['min_rtt_ms'].quantile(.995))

upload_model_space = pd.DataFrame(upload_X, columns=['standardized_log1p_upload', 'standardized_log1p_rtt'])
scatter_one_pair(axes[2], upload_model_space, upload_km.labels_, upload_report,
                 'standardized_log1p_upload', 'standardized_log1p_rtt',
                 'Model space used by K-means', centroid_space='model', km=upload_km)
fig.suptitle(f'Upload K-means assignments — k={upload_k}, silhouette={upload_silhouette:.3f}', y=.99, fontsize=15)
fig.tight_layout(rect=[0, 0, 1, .93]); fig.savefig(FIGURE_DIR / 'upload_cluster_three_views.png', dpi=200, bbox_inches='tight'); plt.show()

### Final presentation table — Upload

In [ ]:
upload_presentation = upload_report[['profile', 'median_upload_mbps', 'median_min_rtt_ms',
                                      'count', 'percent']].rename(columns={
    'median_upload_mbps': 'Throughput (Mbps)', 'median_min_rtt_ms': 'RTT (ms)',
    'count': 'Count', 'percent': 'Percentage', 'profile': 'Profile'})
upload_presentation = upload_presentation.round({'Throughput (Mbps)': 2, 'RTT (ms)': 2, 'Percentage': 2})
upload_presentation.to_csv(DATA_DIR / 'upload_cluster_profiles.csv', index=False)
display(upload_presentation)

upload_colour_map = ranked_cluster_colours(upload_report)
colors = [upload_colour_map[int(cluster)][1] for cluster in upload_report['cluster']]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(upload_report['profile'], upload_report['percent'], color=colors)
for y, (pct, mbps) in enumerate(zip(upload_report['percent'], upload_report['median_upload_mbps'])):
    ax.text(pct + 0.5, y, f'{pct:.1f}%  (median {mbps:.1f} Mbps)', va='center', fontsize=9)
ax.set(title='Upload network-quality profiles', xlabel='% of clean rows')
ax.invert_yaxis(); ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'upload_cluster_profiles.png', dpi=180, bbox_inches='tight'); plt.show()

### Nearest real measurement to each centroid — Upload

In [ ]:
upload_nearest = nearest_rows_to_centroids(upload_X, upload_km, upload)
upload_nearest = upload_nearest.merge(upload_report[['cluster', 'profile']], on='cluster')
upload_nearest_display = upload_nearest[['profile', 'measurement_id', 'upload_mbps', 'min_rtt_ms']]
upload_nearest_display.to_csv(DATA_DIR / 'upload_cluster_nearest_rows.csv', index=False)
display(upload_nearest_display)

In [ ]:
worst_profile_composition(upload_report, upload_working)

### Upload: where each profile is located

Same simple map for upload profiles.

In [ ]:
upload_geo = upload.loc[upload_working.index, ['latitude', 'longitude']].copy()
upload_geo['cluster'] = upload_working['cluster'].values

fig, ax = plt.subplots(figsize=(8, 7))
for cluster, (profile, colour) in ranked_cluster_colours(upload_report).items():
    sub = upload_geo.loc[upload_geo['cluster'] == cluster]
    ax.scatter(sub['longitude'], sub['latitude'], s=6, alpha=.35, color=colour, edgecolors='none',
               label=f'{profile} (n={len(sub):,})')
ax.set(xlim=(67, 99), ylim=(6, 38), title='Upload: network-quality profile by location',
       xlabel='Longitude', ylabel='Latitude')
ax.grid(alpha=.15)
ax.legend(fontsize=9, markerscale=3, loc='lower left')
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'upload_profile_map.png', dpi=180, bbox_inches='tight'); plt.show()